In [1]:
#color-boxes_model_output_verify_1
import os
import cv2
import torch
from ultralytics import YOLO
from pathlib import Path
from typing import Tuple


# =========================
# CONFIGURATION
# =========================

YOLO_WEIGHTS = r"/home/khushi/Pixonate/New_Anemia/Models/Yolo model/Eye model/best1.pt"

CONF_THRESH = 0.7
YOLO_IMGSZ = 1024

SAVE_OVERLAYS = True
SHOW_PLOTS = False

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

INPUT_DIR = "/home/khushi/Pixonate/New_Anemia/R_new_anemia/model_outputs/eye_outputs/Finetune_data/CV_images"
OUTPUT_DIR = "/home/khushi/Pixonate/New_Anemia/R_new_anemia/model_outputs/eye_outputs/Finetune_data/CV_images/results"
OVERLAY_DIR = os.path.join(OUTPUT_DIR, "overlays")
LABEL_DIR = os.path.join(OUTPUT_DIR, "labels")

IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".tif")


# =========================
# UTILITY FUNCTIONS
# =========================

def create_output_dirs() -> None:
    """Create required output directories."""
    os.makedirs(OVERLAY_DIR, exist_ok=True)
    os.makedirs(LABEL_DIR, exist_ok=True)


def load_yolo_model(weights_path: str, device: str) -> YOLO:
    """Load YOLO model on specified device."""
    model = YOLO(weights_path)
    model.to(device)
    return model


def convert_to_yolo_format(
    box: Tuple[float, float, float, float],
    img_width: int,
    img_height: int
) -> Tuple[float, float, float, float]:
    """Convert xyxy box format to normalized YOLO format."""
    x1, y1, x2, y2 = box

    x_center = ((x1 + x2) / 2) / img_width
    y_center = ((y1 + y2) / 2) / img_height
    width = (x2 - x1) / img_width
    height = (y2 - y1) / img_height

    return x_center, y_center, width, height


def save_yolo_annotations(
    result,
    image_shape: Tuple[int, int],
    label_path: str
) -> None:
    """Save YOLO-format annotations to file."""
    h, w = image_shape

    with open(label_path, "w") as f:
        if result.boxes is None:
            return

        for box in result.boxes:
            cls_id = int(box.cls.item())
            x1, y1, x2, y2 = box.xyxy[0].tolist()

            x_c, y_c, bw, bh = convert_to_yolo_format(
                (x1, y1, x2, y2), w, h
            )

            f.write(
                f"{cls_id} "
                f"{x_c:.6f} {y_c:.6f} "
                f"{bw:.6f} {bh:.6f}\n"
            )



def draw_boxes(image, result) -> None:
    """Draw bounding boxes with confidence score on image."""
    if result.boxes is None:
        return

    for box in result.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
        conf = float(box.conf.item())  # confidence score
        cls_id = int(box.cls.item())

        # Draw bounding box
        cv2.rectangle(
            image,
            (x1, y1),
            (x2, y2),
            (0, 255, 0),
            2
        )

        # Label text (class + confidence)
        label = f"ID:{cls_id} {conf:.2f}"

        # Background for text (better visibility)
        (text_w, text_h), _ = cv2.getTextSize(
            label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1
        )

        cv2.rectangle(
            image,
            (x1, y1 - text_h - 6),
            (x1 + text_w + 4, y1),
            (0, 255, 0),
            -1
        )

        # Put text
        cv2.putText(
            image,
            label,
            (x1 + 2, y1 - 4),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (0, 0, 0),
            1,
            cv2.LINE_AA
        )


# =========================
# CORE PROCESSING
# =========================

def process_single_image(model: YOLO, image_path: str) -> None:
    """Run YOLO prediction on a single image."""
    image = cv2.imread(image_path)
    if image is None:
        print(f"⚠️ Unable to read image: {image_path}")
        return

    h, w = image.shape[:2]
    img_name = Path(image_path).name
    img_stem = Path(image_path).stem

    # Run inference
    results = model.predict(
        source=image_path,
        conf=CONF_THRESH,
        imgsz=YOLO_IMGSZ,
        device=DEVICE,
        save=False,
        verbose=False
    )

    result = results[0]

    # Save annotations
    label_path = os.path.join(LABEL_DIR, f"{img_stem}.txt")
    save_yolo_annotations(result, (h, w), label_path)

    # Draw & save overlays
    if SAVE_OVERLAYS:
        draw_boxes(image, result)
        overlay_path = os.path.join(OVERLAY_DIR, img_name)
        cv2.imwrite(overlay_path, image)

    # Optional visualization
    if SHOW_PLOTS:
        cv2.imshow("YOLO Prediction", image)
        cv2.waitKey(0)


def process_image_folder(model: YOLO, folder_path: str) -> None:
    """Process all images in a folder."""
    for file_name in os.listdir(folder_path):
        if file_name.lower().endswith(IMAGE_EXTENSIONS):
            image_path = os.path.join(folder_path, file_name)
            process_single_image(model, image_path)


# =========================
# MAIN ENTRY POINT
# =========================

def main() -> None:
    create_output_dirs()
    model = load_yolo_model(YOLO_WEIGHTS, DEVICE)
    process_image_folder(model, INPUT_DIR)

    if SHOW_PLOTS:
        cv2.destroyAllWindows()
    print("✅ YOLO prediction completed successfully.")
    print(f"📁 Overlays: {OVERLAY_DIR}")
    print(f"📁 Labels:   {LABEL_DIR}")


if __name__ == "__main__":
    main()


KeyboardInterrupt: 

In [ ]:
# #wrongly_predicted_save_in_another_folder_2
import os
import shutil
from PIL import Image, ImageOps
import matplotlib
import matplotlib.pyplot as plt

# FORCE REAL WINDOW BACKEND
matplotlib.use("QtAgg")

# =========================
# CONFIG
# =========================

IMG_DIR = "/home/khushi/Pixonate/New_Anemia/R_new_anemia/overall_main_dataset_found/Conjuctiva"
LBL_DIR = "/home/khushi/Pixonate/New_Anemia/R_new_anemia/model_outputs/eye_outputs1/labels"

OUT_IMG_DIR = "/home/khushi/Pixonate/New_Anemia/R_new_anemia/model_outputs/eye_outputs1/Finetune_data/images"
OUT_LBL_DIR = "/home/khushi/Pixonate/New_Anemia/R_new_anemia/model_outputs/eye_outputs1/Finetune_data/labels"

os.makedirs(OUT_IMG_DIR, exist_ok=True)
os.makedirs(OUT_LBL_DIR, exist_ok=True)

IMAGE_EXTS = (".jpg", ".jpeg", ".png")

# =========================
# UTILS
# =========================

def draw_yolo_boxes(ax, label_path, w, h):
    if not os.path.exists(label_path):
        return

    with open(label_path) as f:
        for line in f:
            _, xc, yc, bw, bh = map(float, line.split())
            x1 = (xc - bw / 2) * w
            y1 = (yc - bh / 2) * h
            bw *= w
            bh *= h

            ax.add_patch(
                plt.Rectangle(
                    (x1, y1),
                    bw,
                    bh,
                    fill=False,
                    edgecolor="lime",
                    linewidth=2
                )
            )


def copy_pair(img_name):
    shutil.copy(
        os.path.join(IMG_DIR, img_name),
        os.path.join(OUT_IMG_DIR, img_name)
    )

    lbl = os.path.splitext(img_name)[0] + ".txt"
    src = os.path.join(LBL_DIR, lbl)

    if os.path.exists(src):
        shutil.copy(src, os.path.join(OUT_LBL_DIR, lbl))


# =========================
# MAIN LOOP
# =========================

for img_name in sorted(os.listdir(IMG_DIR)):
    if not img_name.lower().endswith(IMAGE_EXTS):
        continue

    img_path = os.path.join(IMG_DIR, img_name)
    lbl_path = os.path.join(LBL_DIR, os.path.splitext(img_name)[0] + ".txt")

    # ✅ FIX: normalize orientation
    img = ImageOps.exif_transpose(Image.open(img_path)).convert("RGB")
    w, h = img.size

    fig, ax = plt.subplots(figsize=(7, 7))
    ax.imshow(img)
    ax.axis("off")
    ax.set_title("c = copy | Enter = skip | q = quit")

    draw_yolo_boxes(ax, lbl_path, w, h)

    pressed = {"key": None}

    def on_key(event):
        pressed["key"] = event.key
        plt.close()

    fig.canvas.mpl_connect("key_press_event", on_key)
    plt.show(block=True)

    if pressed["key"] == "c":
        copy_pair(img_name)
    elif pressed["key"] == "q":
        break


In [8]:
#eye_model_output_verify
import os
import cv2
import torch
from ultralytics import YOLO
from pathlib import Path
from typing import Tuple


# =========================
# CONFIGURATION
# =========================

YOLO_WEIGHTS = r"/home/khushi/Pixonate/New_Anemia/Models/Yolo model/Eye model/best1.pt"

CONF_THRESH = 0.7
YOLO_IMGSZ = 1024

SAVE_OVERLAYS = True
SHOW_PLOTS = False

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

INPUT_DIR = "/home/khushi/Pixonate/New_Anemia/R_new_anemia/overall_main_dataset_found/Conjuctiva"
OUTPUT_DIR = "/home/khushi/Pixonate/New_Anemia/R_new_anemia/model_outputs/eye_outputs"
OVERLAY_DIR = os.path.join(OUTPUT_DIR, "overlays")
LABEL_DIR = os.path.join(OUTPUT_DIR, "labels")

IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".tif")


# =========================
# UTILITY FUNCTIONS
# =========================

def create_output_dirs() -> None:
    """Create required output directories."""
    os.makedirs(OVERLAY_DIR, exist_ok=True)
    os.makedirs(LABEL_DIR, exist_ok=True)


def load_yolo_model(weights_path: str, device: str) -> YOLO:
    """Load YOLO model on specified device."""
    model = YOLO(weights_path)
    model.to(device)
    return model


def convert_to_yolo_format(
    box: Tuple[float, float, float, float],
    img_width: int,
    img_height: int
) -> Tuple[float, float, float, float]:
    """Convert xyxy box format to normalized YOLO format."""
    x1, y1, x2, y2 = box

    x_center = ((x1 + x2) / 2) / img_width
    y_center = ((y1 + y2) / 2) / img_height
    width = (x2 - x1) / img_width
    height = (y2 - y1) / img_height

    return x_center, y_center, width, height


def save_yolo_annotations(
    result,
    image_shape: Tuple[int, int],
    label_path: str
) -> None:
    """Save YOLO-format annotations to file."""
    h, w = image_shape

    with open(label_path, "w") as f:
        if result.boxes is None:
            return

        for box in result.boxes:
            cls_id = int(box.cls.item())
            x1, y1, x2, y2 = box.xyxy[0].tolist()

            x_c, y_c, bw, bh = convert_to_yolo_format(
                (x1, y1, x2, y2), w, h
            )

            f.write(
                f"{cls_id} "
                f"{x_c:.6f} {y_c:.6f} "
                f"{bw:.6f} {bh:.6f}\n"
            )



def draw_boxes(image, result) -> None:
    """Draw bounding boxes with confidence score on image."""
    if result.boxes is None:
        return

    for box in result.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
        conf = float(box.conf.item())  # confidence score
        cls_id = int(box.cls.item())

        # Draw bounding box
        cv2.rectangle(
            image,
            (x1, y1),
            (x2, y2),
            (0, 255, 0),
            2
        )

        # Label text (class + confidence)
        label = f"ID:{cls_id} {conf:.2f}"

        # Background for text (better visibility)
        (text_w, text_h), _ = cv2.getTextSize(
            label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1
        )

        cv2.rectangle(
            image,
            (x1, y1 - text_h - 6),
            (x1 + text_w + 4, y1),
            (0, 255, 0),
            -1
        )

        # Put text
        cv2.putText(
            image,
            label,
            (x1 + 2, y1 - 4),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (0, 0, 0),
            1,
            cv2.LINE_AA
        )


# =========================
# CORE PROCESSING
# =========================

def process_single_image(model: YOLO, image_path: str) -> None:
    """Run YOLO prediction on a single image."""
    image = cv2.imread(image_path)
    if image is None:
        print(f"⚠️ Unable to read image: {image_path}")
        return

    h, w = image.shape[:2]
    img_name = Path(image_path).name
    img_stem = Path(image_path).stem

    # Run inference
    results = model.predict(
        source=image_path,
        conf=CONF_THRESH,
        imgsz=YOLO_IMGSZ,
        device=DEVICE,
        save=False,
        verbose=False
    )

    result = results[0]

    # Save annotations
    label_path = os.path.join(LABEL_DIR, f"{img_stem}.txt")
    save_yolo_annotations(result, (h, w), label_path)

    # Draw & save overlays
    if SAVE_OVERLAYS:
        draw_boxes(image, result)
        overlay_path = os.path.join(OVERLAY_DIR, img_name)
        cv2.imwrite(overlay_path, image)

    # Optional visualization
    if SHOW_PLOTS:
        cv2.imshow("YOLO Prediction", image)
        cv2.waitKey(0)


def process_image_folder(model: YOLO, folder_path: str) -> None:
    """Process all images in a folder."""
    for file_name in os.listdir(folder_path):
        if file_name.lower().endswith(IMAGE_EXTENSIONS):
            image_path = os.path.join(folder_path, file_name)
            process_single_image(model, image_path)


# =========================
# MAIN ENTRY POINT
# =========================

def main() -> None:
    create_output_dirs()
    model = load_yolo_model(YOLO_WEIGHTS, DEVICE)
    process_image_folder(model, INPUT_DIR)

    if SHOW_PLOTS:
        cv2.destroyAllWindows()
    print("✅ YOLO prediction completed successfully.")
    print(f"📁 Overlays: {OVERLAY_DIR}")
    print(f"📁 Labels:   {LABEL_DIR}")


if __name__ == "__main__":
    main()


✅ YOLO prediction completed successfully.
📁 Overlays: /home/khushi/Pixonate/New_Anemia/R_new_anemia/model_outputs/eye_outputs/overlays
📁 Labels:   /home/khushi/Pixonate/New_Anemia/R_new_anemia/model_outputs/eye_outputs/labels


In [2]:
#eye_model_with_one_eye_detect_logic
import os
import cv2
import torch
from ultralytics import YOLO
from pathlib import Path
from typing import Tuple, Optional

# =========================
# CONFIG
# =========================

EYE_WEIGHTS = r"/home/khushi/Pixonate/New_Anemia/Models/Yolo model/Eye model/best1.pt"
COLOR_WEIGHTS = r"/home/khushi/Pixonate/New_Anemia/Models/Yolo model/Color boxes model/best1.pt"

CONF_THRESH = 0.75
YOLO_IMGSZ = 1024

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

INPUT_DIR = "/home/khushi/Pixonate/New_Anemia/R_new_anemia/model_outputs/eye_outputs/Finetune_data/CV_images"
OUTPUT_DIR = "/home/khushi/Pixonate/New_Anemia/R_new_anemia/model_outputs/eye_outputs/Finetune_data/new"

OVERLAY_DIR = os.path.join(OUTPUT_DIR, "overlays")
LABEL_DIR = os.path.join(OUTPUT_DIR, "labels")

IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png")

# =========================
# SETUP
# =========================

def setup_dirs():
    os.makedirs(OVERLAY_DIR, exist_ok=True)
    os.makedirs(LABEL_DIR, exist_ok=True)


def load_model(weights: str) -> YOLO:
    model = YOLO(weights)
    model.to(DEVICE)
    return model


# =========================
# GEOMETRY
# =========================

def to_yolo(box, w, h):
    x1, y1, x2, y2 = box
    xc = ((x1 + x2) / 2) / w
    yc = ((y1 + y2) / 2) / h
    bw = (x2 - x1) / w
    bh = (y2 - y1) / h
    return xc, yc, bw, bh


def largest_box(boxes) -> Optional[Tuple[float, float, float, float]]:
    if not boxes:
        return None
    return max(boxes, key=lambda b: (b[2] - b[0]) * (b[3] - b[1]))


def select_eye(eye_dets, palette_box):
    px1, py1, px2, py2 = palette_box
    palette_top = py1
    tolerance = 0.15 * (py2 - py1)

    valid = []
    for box, conf in eye_dets:
        _, y1, _, y2 = box
        center_y = (y1 + y2) / 2

        if center_y < palette_top + tolerance:
            valid.append((box, conf, abs(center_y - palette_top)))

    if not valid:
        return None, None

    valid.sort(key=lambda x: x[2])
    return valid[0][0], valid[0][1]


# =========================
# PROCESSING
# =========================

def process_image(eye_model, color_model, img_path):
    img = cv2.imread(img_path)
    if img is None:
        return

    h, w = img.shape[:2]
    name = Path(img_path).stem

    eye_res = eye_model.predict(
        img_path, conf=CONF_THRESH, imgsz=YOLO_IMGSZ,
        device=DEVICE, verbose=False
    )[0]

    color_res = color_model.predict(
        img_path, conf=CONF_THRESH, imgsz=YOLO_IMGSZ,
        device=DEVICE, verbose=False
    )[0]

    eye_dets = []
    if eye_res.boxes:
        for b in eye_res.boxes:
            eye_dets.append((b.xyxy[0].tolist(), float(b.conf.item())))

    palette_boxes = [b.xyxy[0].tolist() for b in color_res.boxes] if color_res.boxes else []

    palette_box = largest_box(palette_boxes)
    if not palette_box or not eye_dets:
        return

    selected_eye, conf = select_eye(eye_dets, palette_box)
    if selected_eye is None:
        return

    # SAVE LABEL
    xc, yc, bw, bh = to_yolo(selected_eye, w, h)
    with open(os.path.join(LABEL_DIR, f"{name}.txt"), "w") as f:
        f.write(f"0 {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}\n")

    # DRAW SELECTED EYE + CONF
    x1, y1, x2, y2 = map(int, selected_eye)
    cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 3)

    label = f"{conf:.2f}"
    (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
    cv2.rectangle(img, (x1, y1 - th - 6), (x1 + tw + 4, y1), (0, 255, 0), -1)
    cv2.putText(
        img, label, (x1 + 2, y1 - 4),
        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 2
    )

    cv2.imwrite(os.path.join(OVERLAY_DIR, f"{name}.jpg"), img)


def process_folder(eye_model, color_model):
    for file in os.listdir(INPUT_DIR):
        if file.lower().endswith(IMAGE_EXTENSIONS):
            process_image(
                eye_model,
                color_model,
                os.path.join(INPUT_DIR, file)
            )


# =========================
# MAIN
# =========================

def main():
    setup_dirs()
    eye_model = load_model(EYE_WEIGHTS)
    color_model = load_model(COLOR_WEIGHTS)

    process_folder(eye_model, color_model)

    print("✅ Done")
    print("📁 Overlays:", OVERLAY_DIR)
    print("📁 Labels:", LABEL_DIR)


if __name__ == "__main__":
    main()


✅ Done
📁 Overlays: /home/khushi/Pixonate/New_Anemia/R_new_anemia/model_outputs/eye_outputs/Finetune_data/new/overlays
📁 Labels: /home/khushi/Pixonate/New_Anemia/R_new_anemia/model_outputs/eye_outputs/Finetune_data/new/labels


In [21]:
import os
import cv2
import torch
import numpy as np
from ultralytics import YOLO
from pathlib import Path
from PIL import Image, ImageOps

import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp

# =========================
# CONFIG
# =========================

YOLO_WEIGHTS = r"/home/khushi/Pixonate/New_Anemia/Models/Yolo model/Eye model/best1.pt"
SEG_WEIGHTS  = r"/home/khushi/Pixonate/New_Anemia/Models/Unet model/best_model_1.pth"

INPUT_DIR  = "/home/khushi/Pixonate/New_Anemia/R_new_anemia/overall_main_dataset_found/Conjuctiva"
OUTPUT_DIR = "/home/khushi/Pixonate/New_Anemia/R_new_anemia/model_outputs/seg_outputs1"

CONF_THRESH = 0.7
YOLO_IMGSZ  = 1024
NET_SIZE    = 256
ALPHA       = 0.45

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

OVERLAY_DIR = os.path.join(OUTPUT_DIR, "overlays")
MASK_DIR    = os.path.join(OUTPUT_DIR, "masks")

os.makedirs(OVERLAY_DIR, exist_ok=True)
os.makedirs(MASK_DIR, exist_ok=True)

# =========================
# LOAD MODELS
# =========================

def load_yolo():
    model = YOLO(YOLO_WEIGHTS)
    model.to(DEVICE)
    return model


def load_segmentation():
    model = smp.Unet(
        encoder_name="resnet34",
        encoder_weights=None,
        in_channels=3,
        classes=1
    )
    state_dict = torch.load(SEG_WEIGHTS, map_location=DEVICE)
    model.load_state_dict(state_dict)
    model.to(DEVICE)
    model.eval()
    return model


# =========================
# SEG TRANSFORM
# =========================

def get_seg_transform():
    return A.Compose([
        A.LongestMaxSize(max_size=NET_SIZE),
        A.PadIfNeeded(
            min_height=NET_SIZE,
            min_width=NET_SIZE,
            border_mode=cv2.BORDER_REFLECT
        ),
        A.Normalize(mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])


# =========================
# SEGMENT CROP
# =========================

def segment_crop(model, crop, transform):
    h, w = crop.shape[:2]

    data = transform(image=crop)
    inp = data["image"].unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        prob = torch.sigmoid(model(inp))[0, 0].cpu().numpy()

    scale = NET_SIZE / max(h, w)
    nw, nh = int(w * scale), int(h * scale)
    px = (NET_SIZE - nw) // 2
    py = (NET_SIZE - nh) // 2

    prob = prob[py:py + nh, px:px + nw]
    mask = (prob > 0.5).astype(np.uint8)

    return cv2.resize(mask, (w, h), interpolation=cv2.INTER_NEAREST)


# =========================
# PROCESS IMAGE
# =========================

def process_image(yolo, seg_model, transform, img_path):
    name = Path(img_path).stem

    img = ImageOps.exif_transpose(Image.open(img_path)).convert("RGB")
    img = np.array(img)
    H, W = img.shape[:2]

    res = yolo.predict(
        img_path,
        conf=CONF_THRESH,
        imgsz=YOLO_IMGSZ,
        device=DEVICE,
        verbose=False
    )[0]

    if not res.boxes:
        return

    # take largest eye box
    boxes = res.boxes.xyxy.cpu().numpy()
    areas = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
    idx = np.argmax(areas)
    x1, y1, x2, y2 = boxes[idx].astype(int)

    x1, y1 = max(0, x1), max(0, y1)
    x2, y2 = min(W, x2), min(H, y2)

    crop = img[y1:y2, x1:x2]
    if crop.size == 0:
        return

    # SEGMENTATION
    mask_local = segment_crop(seg_model, crop, transform)

    # Segmentation confidence = mean probability over the predicted mask area
    seg_conf = mask_local.mean()  # value between 0 and 1

    full_mask = np.zeros((H, W), dtype=np.uint8)
    full_mask[y1:y2, x1:x2] = mask_local

    # light overlay for visualization
    overlay = img.copy()
    overlay[full_mask == 1] = (
        (1 - ALPHA) * overlay[full_mask == 1] +
        ALPHA * np.array([0, 255, 0])
    ).astype(np.uint8)

    # Draw segmentation confidence on overlay
    cv2.putText(
        overlay,
        f"Seg Conf: {seg_conf:.2f}",
        (10, 30),  # fixed position
        cv2.FONT_HERSHEY_SIMPLEX,
        1.0,        # fixed font size
        (0, 255, 0),
        2
    )

    # Save results
    cv2.imwrite(
        os.path.join(OVERLAY_DIR, f"{name}.jpg"),
        cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR)
    )
    cv2.imwrite(
        os.path.join(MASK_DIR, f"{name}.png"),
        full_mask * 255
    )


# =========================
# MAIN
# =========================

def main():
    yolo = load_yolo()
    seg_model = load_segmentation()
    transform = get_seg_transform()

    for img_name in sorted(os.listdir(INPUT_DIR)):
        if img_name.lower().endswith((".jpg", ".jpeg", ".png")):
            process_image(
                yolo,
                seg_model,
                transform,
                os.path.join(INPUT_DIR, img_name)
            )

    print("✅ Eye segmentation completed")
    print("📁 Overlays:", OVERLAY_DIR)
    print("📁 Masks:", MASK_DIR)


if __name__ == "__main__":
    main()


✅ Eye segmentation completed
📁 Overlays: /home/khushi/Pixonate/New_Anemia/R_new_anemia/model_outputs/seg_outputs1/overlays
📁 Masks: /home/khushi/Pixonate/New_Anemia/R_new_anemia/model_outputs/seg_outputs1/masks


In [29]:
# =========================
# SEGMENTATION CHECK TOOL
# =========================

import os
import cv2
import shutil
import numpy as np
from pathlib import Path
from PIL import Image, ImageOps

import matplotlib
import matplotlib.pyplot as plt

# FORCE REAL WINDOW
matplotlib.use("QtAgg")

# =========================
# CONFIG
# =========================

IMG_DIR = "/home/khushi/Pixonate/New_Anemia/R_new_anemia/overall_main_dataset_found/Conjuctiva"
MASK_DIR = "/home/khushi/Pixonate/New_Anemia/R_new_anemia/model_outputs/seg_outputs1/masks"

# where wrongly segmented samples will go
OUT_IMG_DIR  = "/home/khushi/Pixonate/New_Anemia/R_new_anemia/model_outputs/seg_outputs1/Finetune_data/images"
OUT_MASK_DIR = "/home/khushi/Pixonate/New_Anemia/R_new_anemia/model_outputs/seg_outputs1/Finetune_data/masks"

IMAGE_EXTS = (".jpg", ".jpeg", ".png")
ALPHA = 0.45   # overlay strength

os.makedirs(OUT_IMG_DIR, exist_ok=True)
os.makedirs(OUT_MASK_DIR, exist_ok=True)

# =========================
# UTILS
# =========================

def load_image(img_path):
    """Load image with correct orientation"""
    img = ImageOps.exif_transpose(Image.open(img_path)).convert("RGB")
    return np.array(img)


def load_mask(mask_path):
    """Load binary mask"""
    if not os.path.exists(mask_path):
        return None
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    return (mask > 0).astype(np.uint8)


def overlay_mask(image, mask):
    """Create green segmentation overlay"""
    overlay = image.copy()
    overlay[mask == 1] = (
        (1 - ALPHA) * overlay[mask == 1] +
        ALPHA * np.array([0, 255, 0])
    ).astype(np.uint8)
    return overlay


def copy_pair(img_name):
    """Copy image + mask to wrong folder"""
    shutil.copy(
        os.path.join(IMG_DIR, img_name),
        os.path.join(OUT_IMG_DIR, img_name)
    )

    mask_name = Path(img_name).stem + ".png"
    src_mask = os.path.join(MASK_DIR, mask_name)

    if os.path.exists(src_mask):
        shutil.copy(
            src_mask,
            os.path.join(OUT_MASK_DIR, mask_name)
        )


# =========================
# MAIN LOOP
# =========================

for img_name in sorted(os.listdir(IMG_DIR)):
    if not img_name.lower().endswith(IMAGE_EXTS):
        continue

    img_path = os.path.join(IMG_DIR, img_name)
    mask_path = os.path.join(MASK_DIR, Path(img_name).stem + ".png")

    image = load_image(img_path)
    mask = load_mask(mask_path)

    if mask is None:
        continue

    overlay = overlay_mask(image, mask)

    fig, ax = plt.subplots(figsize=(7, 7))
    ax.imshow(overlay)
    ax.axis("off")
    ax.set_title("c = copy (wrong) | Enter = skip | q = quit")

    pressed = {"key": None}

    def on_key(event):
        pressed["key"] = event.key
        plt.close()

    fig.canvas.mpl_connect("key_press_event", on_key)
    plt.show(block=True)

    if pressed["key"] == "c":
        copy_pair(img_name)

    elif pressed["key"] == "q":
        break

print("✅ Segmentation checking finished")
print("❌ Wrong samples saved to:", OUT_IMG_DIR)


✅ Segmentation checking finished
❌ Wrong samples saved to: /home/khushi/Pixonate/New_Anemia/R_new_anemia/model_outputs/seg_outputs1/Finetune_data/images


In [3]:
import os
import cv2
import torch
import numpy as np
from ultralytics import YOLO

In [5]:
import os
from ultralytics import YOLO
from pathlib import Path
import cv2

# =========================
# CONFIG (CHANGE THESE)
# =========================
MODEL_PATH = "/home/khushi/Pixonate/New_Anemia/Models/Yolo model/Eye model/best.pt"
INPUT_DIR = "/home/khushi/Pixonate/New_Anemia/R_new_anemia/model_outputs/eye_outputs/Finetune_data/images"
OUTPUT_DIR = "/home/khushi/Pixonate/New_Anemia/R_new_anemia/model_outputs/eye_outputs/Finetune_data/img_res"
IMG_EXTS = [".jpg", ".jpeg", ".png"]

# =========================
# LOAD MODEL
# =========================
model = YOLO(MODEL_PATH)

# create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

# =========================
# RUN PREDICTION
# =========================
for img_name in os.listdir(INPUT_DIR):
    if Path(img_name).suffix.lower() not in IMG_EXTS:
        continue

    img_path = os.path.join(INPUT_DIR, img_name)

    # run inference
    results = model(img_path, conf=0.7)

    # get plotted image
    annotated_img = results[0].plot()

    # save output
    out_path = os.path.join(OUTPUT_DIR, img_name)
    cv2.imwrite(out_path, annotated_img)

    print(f"Saved: {out_path}")

print("✅ Prediction completed.")



image 1/1 /home/khushi/Pixonate/New_Anemia/R_new_anemia/model_outputs/eye_outputs/Finetune_data/images/1400.12.9.jpg: 1024x768 1 0, 124.8ms
Speed: 3.5ms preprocess, 124.8ms inference, 0.6ms postprocess per image at shape (1, 3, 1024, 768)
Saved: /home/khushi/Pixonate/New_Anemia/R_new_anemia/model_outputs/eye_outputs/Finetune_data/img_res/1400.12.9.jpg

image 1/1 /home/khushi/Pixonate/New_Anemia/R_new_anemia/model_outputs/eye_outputs/Finetune_data/images/317.13.9.jpg: 1024x800 1 0, 116.2ms
Speed: 3.9ms preprocess, 116.2ms inference, 0.6ms postprocess per image at shape (1, 3, 1024, 800)
Saved: /home/khushi/Pixonate/New_Anemia/R_new_anemia/model_outputs/eye_outputs/Finetune_data/img_res/317.13.9.jpg

image 1/1 /home/khushi/Pixonate/New_Anemia/R_new_anemia/model_outputs/eye_outputs/Finetune_data/images/1791.11.1.jpg: 1024x768 1 0, 98.8ms
Speed: 5.0ms preprocess, 98.8ms inference, 0.7ms postprocess per image at shape (1, 3, 1024, 768)
Saved: /home/khushi/Pixonate/New_Anemia/R_new_anemia/m

In [ ]:
# model_prediction_geojson_1
import os
import json
from ultralytics import YOLO
from pathlib import Path
import cv2

# =========================
# CONFIG
# =========================
MODEL_PATH = "/home/khushi/Pixonate/New_Anemia/Models/Yolo model/Color boxes model/best1.pt"
INPUT_DIR = "/home/khushi/Pixonate/New_Anemia/R_new_anemia/model_outputs/color_box_outputs/Data/CV_images"
OUTPUT_IMG_DIR = "/home/khushi/Pixonate/New_Anemia/R_new_anemia/model_outputs/color_box_outputs/Data/model_prediction"
OUTPUT_GEOJSON_DIR = "/home/khushi/Pixonate/New_Anemia/R_new_anemia/model_outputs/color_box_outputs/Data/CV_images/geojson"

IMG_EXTS = [".jpg", ".jpeg", ".png"]
CONF_THRES = 0.7

# =========================
# LOAD MODEL
# =========================
model = YOLO(MODEL_PATH)

os.makedirs(OUTPUT_IMG_DIR, exist_ok=True)
os.makedirs(OUTPUT_GEOJSON_DIR, exist_ok=True)

# =========================
# RUN PREDICTION
# =========================
for img_name in os.listdir(INPUT_DIR):
    if Path(img_name).suffix.lower() not in IMG_EXTS:
        continue

    img_path = os.path.join(INPUT_DIR, img_name)

    results = model(img_path, conf=CONF_THRES)
    r = results[0]

    # -------------------------
    # Save annotated image
    # -------------------------
    annotated_img = r.plot()
    cv2.imwrite(os.path.join(OUTPUT_IMG_DIR, img_name), annotated_img)

    # -------------------------
    # Create GeoJSON
    # -------------------------
    features = []

    if r.boxes is not None:
        for box in r.boxes:
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            cls_id = int(box.cls[0])
            conf = float(box.conf[0])
            class_name = model.names[cls_id]

            feature = {
                "type": "Feature",
                "properties": {
                    "class_id": cls_id,
                    "class_name": class_name,
                    "confidence": conf
                },
                "geometry": {
                    "type": "Polygon",
                    "coordinates": [[
                        [x1, y1],
                        [x2, y1],
                        [x2, y2],
                        [x1, y2],
                        [x1, y1]
                    ]]
                }
            }
            features.append(feature)

    geojson_data = {
        "type": "FeatureCollection",
        "features": features
    }

    geojson_path = os.path.join(
        OUTPUT_GEOJSON_DIR,
        Path(img_name).stem + ".geojson"
    )

    with open(geojson_path, "w") as f:
        json.dump(geojson_data, f, indent=2)

    print(f"Saved image + geojson for: {img_name}")

print("✅ Prediction + GeoJSON export completed.")



image 1/1 /home/khushi/Pixonate/New_Anemia/R_new_anemia/model_outputs/color_box_outputs/Data/CV_images/1864.7.5.jpg: 640x480 23 0s, 103.1ms
Speed: 1.9ms preprocess, 103.1ms inference, 0.7ms postprocess per image at shape (1, 3, 640, 480)
Saved image + geojson for: 1864.7.5.jpg

image 1/1 /home/khushi/Pixonate/New_Anemia/R_new_anemia/model_outputs/color_box_outputs/Data/CV_images/1156.10.7.jpg: 640x480 24 0s, 74.3ms
Speed: 1.9ms preprocess, 74.3ms inference, 0.7ms postprocess per image at shape (1, 3, 640, 480)
Saved image + geojson for: 1156.10.7.jpg

image 1/1 /home/khushi/Pixonate/New_Anemia/R_new_anemia/model_outputs/color_box_outputs/Data/CV_images/1548.14.3.jpg: 640x480 24 0s, 75.5ms
Speed: 2.0ms preprocess, 75.5ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 480)
Saved image + geojson for: 1548.14.3.jpg

image 1/1 /home/khushi/Pixonate/New_Anemia/R_new_anemia/model_outputs/color_box_outputs/Data/CV_images/1149.11.7.jpg: 640x480 24 0s, 88.3ms
Speed: 1.9ms preproces